# README
- **Author**: RUEI-HSIANG, CHANG（張睿庠；B12303030；AI協助：Codex）
- **Created At**: 2026-11-14
- **Last Modified At**: 2026-11-14

## What does this file do?
對兩黨差距向量 `[光譜差距, 黨員規模差距]` 乘以單位向量 `u=(1,k)/sqrt(1+k²)`（內積），k=3、5、10，比例依序為光譜：規模。每組結果獨立以441組配對的min–max轉為0–1，包含自身配對；此資料raw最小值0，最大值由實際配對取得。不是將權重除以總和，也不是先投影各黨位置再取絕對差。

## What does this file take?
- Source Data Sets: `data/processed/party_difference_vector_20261114.csv`；鍵party_a+party_b；21×21=441列。
- Documentation: `data/processed/party_vector_20261114_documentation.ipynb`。
- SHA-256: `4d56330bc5628eac95054c12086baa388612e86421c4b6f4abf5e56bb15a22de`。
- 沿用來源的100人假設、新黨2000近似、不同年度／口徑及無黨籍／難歸類光譜距離特例。

## What does this file output?
- `data/processed/party_weighted_distances_20261114.csv`：441列，party_a/b、原差距兩欄，以及每組raw與0_1欄，總10欄。
- Role: analysis input；Documentation: `data/processed/party_weighted_distances_20261114_documentation.ipynb`。
- Verification: 執行輸出保留重讀、鍵、對稱、對角線與範圍檢查及SHA-256。
- 未改原資料、未重跑分群。重新執行前需檢查既有輸出，程式不直接覆蓋。


In [1]:
from pathlib import Path
import csv, math, hashlib
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists())
SOURCE=ROOT/'data/processed/party_difference_vector_20261114.csv'
assert hashlib.sha256(SOURCE.read_bytes()).hexdigest()=='4d56330bc5628eac95054c12086baa388612e86421c4b6f4abf5e56bb15a22de'
with SOURCE.open(encoding='utf-8-sig',newline='') as f: rows=list(csv.DictReader(f))
assert len(rows)==441 and len({(r['party_a'],r['party_b']) for r in rows})==441
result=[]
for r in rows:
    a=float(r['party_spectrum_difference']);b=float(r['party_size_difference'])
    item={'party_a':r['party_a'],'party_b':r['party_b'],'party_spectrum_difference':a,'party_size_difference':b}
    for k in [3,5,10]: item[f'distance_1_{k}_raw']=(a+k*b)/math.sqrt(1+k*k)
    result.append(item)
for k in [3,5,10]:
    raw=f'distance_1_{k}_raw'; normalized=f'distance_1_{k}_0_1'
    lo=min(r[raw] for r in result);hi=max(r[raw] for r in result)
    assert hi>lo
    for r in result: r[normalized]=(r[raw]-lo)/(hi-lo)
    print('ratio',f'1:{k}','unit vector',(1/math.sqrt(1+k*k),k/math.sqrt(1+k*k)),'raw range',(lo,hi))
OUTPUT=ROOT/'data/processed/party_weighted_distances_20261114.csv'
assert OUTPUT!=SOURCE
assert not OUTPUT.exists(), 'Output already exists: inspect before replacing'
with OUTPUT.open('x',encoding='utf-8-sig',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(result[0]));w.writeheader();w.writerows(result)
with OUTPUT.open(encoding='utf-8-sig',newline='') as f: saved=list(csv.DictReader(f))
assert len(saved)==441
assert all(saved[i]=={k:str(v) for k,v in r.items()} for i,r in enumerate(result))
pairs={(r['party_a'],r['party_b']):r for r in saved}
for k in [3,5,10]:
    col=f'distance_1_{k}_0_1'
    assert min(float(r[col]) for r in saved)==0 and max(float(r[col]) for r in saved)==1
    assert all(r[col]==pairs[b,a][col] for (a,b),r in pairs.items())
    assert all(float(r[col])==0 for (a,b),r in pairs.items() if a==b)
print('Verified: 441 rows, 21 parties, three symmetric 0–1 distances, diagonal zero, CSV exact read-back')
print('SHA-256',hashlib.sha256(OUTPUT.read_bytes()).hexdigest())
for a,b in [('中國國民黨','民主進步黨'),('民主進步黨','台灣民眾黨')]:print(pairs[a,b])


ratio 1:3 unit vector (0.31622776601683794, 0.9486832980505138) raw range (0.0, 1.2649110640673518)
ratio 1:5 unit vector (0.19611613513818404, 0.9805806756909202) raw range (0.0, 1.1766968108291043)
ratio 1:10 unit vector (0.09950371902099892, 0.9950371902099892) raw range (0.0, 1.0945409092309881)
Verified: 441 rows, 21 parties, three symmetric 0–1 distances, diagonal zero, CSV exact read-back
SHA-256 bc7454bd77c457179736afcbacdbcff58cf6145558788c2de3bbb1c51c1abd29
{'party_a': '中國國民黨', 'party_b': '民主進步黨', 'party_spectrum_difference': '1.0', 'party_size_difference': '0.2799372189188374', 'distance_1_3_raw': '0.5817995301078492', 'distance_1_5_raw': '0.47061716241665463', 'distance_1_10_raw': '0.3780516627691975', 'distance_1_3_0_1': '0.459952914189128', 'distance_1_5_0_1': '0.39994768243236445', 'distance_1_10_0_1': '0.3453974717443976'}
{'party_a': '民主進步黨', 'party_b': '台灣民眾黨', 'party_spectrum_difference': '0.6', 'party_size_difference': '0.6221303311098366', 'distance_1_3_raw': '0.77